In [1]:
!nvidia-smi
print("GPU 상태 확인 완료!")

Mon Jul 27 00:34:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# =====================================================================
# 1단계: YOLO 위치 검출 모델 학습 (Google Colab에서 실행 권장)
# =====================================================================
# 사용 순서
# 1) https://colab.research.google.com 접속 → 새 노트북
# 2) 상단 메뉴 [런타임] > [런타임 유형 변경] > 하드웨어 가속기 = GPU 선택
# 3) 이 파일 내용을 셀에 붙여넣고 위에서부터 순서대로 실행
# 4) ROBOFLOW_API_KEY, WORKSPACE, PROJECT, VERSION 은 본인 Roboflow
#    프로젝트 페이지 우측 상단 "Download Dataset" 버튼 눌렀을 때 나오는
#    코드에서 그대로 복사하면 됩니다.
# =====================================================================

# --- 설치 ---
!pip install ultralytics roboflow -q
# %pip install ultralytics roboflow -q

# --- 1) Roboflow에서 바운딩박스 라벨 포함 데이터셋 다운로드 ---
from roboflow import Roboflow

ROBOFLOW_API_KEY = "08pEqA03ywLShGQ8Vk09"
WORKSPACE = "s-workspace-ntur3"
PROJECT = "1trashset"
VERSION = 2  # Roboflow 프로젝트 버전 번호

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

# 사용 가능한 버전 번호 확인 (VERSION 값이 실제 존재하는지 먼저 체크)
print("사용 가능한 버전:", [v.version for v in project.versions()])

dataset = project.version(VERSION).download("yolov8")
# 다운로드된 폴더 안에 data.yaml (클래스 이름 정의) + train/valid/test 가 생김

print("데이터셋 위치:", dataset.location)

# --- 2) YOLOv8n(nano)으로 전이학습 ---
# nano 버전을 쓰는 이유: Jetson Nano처럼 연산이 약한 보드에 올리기엔
# 가장 가벼운 버전이 안전합니다. (s/m/l/x 로 갈수록 무겁고 정확하지만 느려짐)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="waste_yolo",
    patience=20,       # 20 epoch 동안 성능 개선 없으면 조기 종료
)

# --- 3) 학습 결과 확인 ---
# 학습이 끝나면 다음 경로에 결과가 저장됩니다.
#   runs/detect/waste_yolo/weights/best.pt   <- 최종 모델 (이걸 사용)
#   runs/detect/waste_yolo/confusion_matrix.png  <- 클래스별 오분류 확인
#   runs/detect/waste_yolo/results.png           <- 학습 곡선(mAP, loss 등)
#
# best.pt 를 다운로드해서 로컬에 저장해두세요.
# (Colab 왼쪽 파일 탐색기에서 우클릭 > 다운로드)

# --- 4) 학습된 모델로 실제 이미지 테스트 (선택) ---
# best_model = YOLO("runs/detect/waste_yolo/weights/best.pt")
# results = best_model.predict("테스트할_이미지_경로.jpg", save=True, conf=0.5)

# =====================================================================
# 다음 단계: best.pt 로 원본 학습 이미지들의 바운딩박스를 잘라내서
# CNN 분류기용 데이터셋을 만듭니다. -> 2_crop_bboxes_for_cnn.py 참고
# =====================================================================

loading Roboflow workspace...
loading Roboflow project...
사용 가능한 버전: ['2', '1']



Extracting Dataset Version Zip to 1trashset-2 in yolov8:: 100%|██████████| 1209/1209 [00:00<00:00, 1477.49it/s]


데이터셋 위치: D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\1trashset-2


100%|██████████| 6.25M/6.25M [00:00<00:00, 97.4MB/s]

New https://pypi.org/project/ultralytics/8.4.107 available  Update with 'pip install -U ultralytics'


...

     81/100         0G     0.7677     0.7246       1.34         18        640: 100%|██████████| 27/27 [01:55<00:00,  4.27s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:15<00:00,  3.84s/it]

                   all        122        116      0.826      0.863      0.861      0.499
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 61, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



81 epochs completed in 3.124 hours.
Optimizer stripped from runs\detect\waste_yolo\weights\last.pt, 5.6MB
Optimizer stripped from runs\detect\waste_yolo\weights\best.pt, 5.6MB

Validating runs\detect\waste_yolo\weights\best.pt...
Ultralytics 8.3.0  Python-3.8.10 torch-2.4.1+cpu CPU (Intel Core(TM) i7-9700 3.00GHz)
Model summary (fused): 186 layers, 2,684,953 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:14<00:00,  3.60s/it]


                   all        122        116      0.827      0.814      0.864      0.526
                 metal         35         35      0.802      0.697      0.836      0.504
                 paper         37         37      0.911      0.973      0.964      0.726
               plastic         44         44      0.768      0.773      0.792      0.349
Speed: 1.3ms preprocess, 37.3ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to runs\detect\waste_yolo


In [ ]:
# # --- 3-1) best.pt를 Google Drive에 백업 (런타임 끊겨도 안전하게 보관) ---
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil, os

# SAVE_DIR = "/content/drive/MyDrive/waste_yolo_results"
# os.makedirs(SAVE_DIR, exist_ok=True)

# shutil.copy("/content/runs/detect/waste_yolo/weights/best.pt", f"{SAVE_DIR}/best.pt")
# shutil.copy("/content/runs/detect/waste_yolo/weights/last.pt", f"{SAVE_DIR}/last.pt")

# print(f"저장 완료: {SAVE_DIR}/best.pt")


#===========================================================================
# # --- 3-1) 결과 폴더를 통째로 압축해서 다운로드 ---
# # 로컬에서 돌렸을 때와 똑같이 first_test/runs/detect/waste_yolo/ 구조가 되도록,
# # 압축 파일을 풀면 그대로 first_test/ 밑에 넣을 수 있는 형태로 만듭니다.
# import shutil

# shutil.make_archive("/content/runs", "zip", "/content", "runs")

# from google.colab import files
# files.download("/content/runs.zip")

# print("runs.zip 다운로드 완료.")
# print("압축 풀어서 나온 runs 폴더를 first_test/ 안에 그대로 넣으면")
# print("로컬에서 돌린 것과 동일하게 first_test/runs/detect/waste_yolo/... 경로가 됩니다.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

runs.zip 다운로드 완료.
압축 풀어서 나온 runs 폴더를 first_test/ 안에 그대로 넣으면
로컬에서 돌린 것과 동일하게 first_test/runs/detect/waste_yolo/... 경로가 됩니다.


In [ ]:
# --- 3-1) Google Drive에 저장 + 자동 다운로드를 위한 공유 설정 ---
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

SAVE_DIR = "/content/drive/MyDrive/waste_yolo_results"
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy("/content/runs/detect/waste_yolo/weights/best.pt", f"{SAVE_DIR}/best.pt")
shutil.copy("/content/runs/detect/waste_yolo/weights/last.pt", f"{SAVE_DIR}/last.pt")

# 폴더를 "링크가 있는 사람은 보기 가능"으로 공유 설정 (로컬 스크립트가 인증 없이 받아갈 수 있도록)
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

result = drive_service.files().list(
    q="name='waste_yolo_results' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive', fields='files(id, name)'
).execute()
folder_id = result['files'][0]['id']

drive_service.permissions().create(
    fileId=folder_id,
    body={'type': 'anyone', 'role': 'reader'},
).execute()

print("저장 완료:", SAVE_DIR)
print("폴더 ID (한 번만 복사해서 first_test/pull_results.py의 FOLDER_ID에 붙여넣으세요):")
print(folder_id)

저장 완료: /content/drive/MyDrive/waste_yolo_results
폴더 ID (한 번만 복사해서 first_test/pull_results.py의 FOLDER_ID에 붙여넣으세요):
1NVM42UWV7zxyu_MisV7o4fllAnvL7d6a
